# Étape 1 - Vérification de l'environnement

Versions, mode d'exécution, chemins, lecture d'un fichier de test et écriture d'un résultat de test dans `outputs/`.

Le fichier source `data/samples/magasins_test.csv` n'est jamais modifié.

In [1]:
import os

# Racine du projet : on remonte depuis le dossier du notebook
RACINE = os.getcwd()
# Racine du depot = dossier qui contient config/parametres.json
while not os.path.exists(os.path.join(RACINE, "config", "parametres.json")):
    if os.path.dirname(RACINE) == RACINE:
        raise RuntimeError("config/parametres.json introuvable : lancer le notebook depuis le depot")
    RACINE = os.path.dirname(RACINE)

# Sans le prefixe file://, Spark irait chercher les fichiers dans HDFS
# (fs.defaultFS = hdfs://localhost:9000 dans la configuration Hadoop)
LOCAL = "file://"

FICHIER_TEST = os.path.join(RACINE, "data", "samples", "magasins_test.csv")
SORTIE_TEST = os.path.join(RACINE, "outputs", "test_environnement")

## 1. Versions Python et PySpark

In [2]:
import platform

import pyspark

print("Version Python  :", platform.python_version())
print("Version PySpark :", pyspark.__version__)

Version Python  : 3.14.4
Version PySpark : 4.3.0


## 2. Session Spark, mode d'exécution et ressources

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP3-Verification-Environnement")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext

print("Version Spark   :", spark.version)
print("Mode d'execution:", sc.master)
print("FS par defaut   :", sc._jsc.hadoopConfiguration().get("fs.defaultFS"))
print("Coeurs utilises :", sc.defaultParallelism)
print("Memoire driver  :", sc.getConf().get("spark.driver.memory", "1g (defaut)"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 14:15:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version Spark   : 4.3.0
Mode d'execution: local[*]
FS par defaut   : hdfs://localhost:9000
Coeurs utilises : 8
Memoire driver  : 1g (defaut)


## 3. Chemins utilisés

In [4]:
print("Racine projet   :", RACINE)
print("Fichier de test :", FICHIER_TEST)
print("Sortie de test  :", SORTIE_TEST)

Racine projet   : /home/lolorigolo/DEV/J1_Retail_Intelligence/tp3_lolo/retail-intelligence-platform
Fichier de test : /home/lolorigolo/DEV/J1_Retail_Intelligence/tp3_lolo/retail-intelligence-platform/data/samples/magasins_test.csv
Sortie de test  : /home/lolorigolo/DEV/J1_Retail_Intelligence/tp3_lolo/retail-intelligence-platform/outputs/test_environnement


## 4. Lecture d'un fichier de test et affichage du schéma

In [5]:
df = spark.read.csv(LOCAL + FICHIER_TEST, header=True, inferSchema=True)
df.printSchema()
df.show()
print("Nombre de lignes:", df.count())

root
 |-- magasin: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- chiffre_affaires: double (nullable = true)



+---------+-----+----------------+
|  magasin|ville|chiffre_affaires|
+---------+-----+----------------+
|Magasin A|Paris|          1200.0|
|Magasin B| Lyon|           850.0|
|Magasin C|Lille|          1600.0|
+---------+-----+----------------+



Nombre de lignes: 3


## 5. Écriture d'un résultat de test dans `outputs/`

In [6]:
df.write.mode("overwrite").parquet(LOCAL + SORTIE_TEST)
relu = spark.read.parquet(LOCAL + SORTIE_TEST)
print("Relecture Parquet OK :", relu.count(), "lignes")

Relecture Parquet OK : 3 lignes


In [2]:
spark.stop()
print("arret de spark")

NameError: name 'spark' is not defined